In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt


In [3]:
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "data/training_set/training_set",
    image_size=(224, 224),
    batch_size=32
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    "data/test_set/test_set",
    image_size=(224, 224),
    batch_size=32
)


Found 8005 files belonging to 2 classes.
Found 2023 files belonging to 2 classes.


In [4]:
train_ds = train_ds.map(lambda x, y: (x/255.0, y))
val_ds   = val_ds.map(lambda x, y: (x/255.0, y))


In [5]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))


In [6]:
model = models.Sequential([
    tf.keras.Input(shape=(224,224,3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(2, activation='softmax')
])


In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)


In [9]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    "best_from_scratch.keras",
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)


In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[checkpoint]
)


Epoch 1/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 710ms/step - accuracy: 0.5478 - loss: 0.6840
Epoch 1: val_accuracy improved from -inf to 0.66387, saving model to best_from_scratch.keras
251/251 ━━━━━━━━━━━━━━━━━━━━ 190s 753ms/step - accuracy: 0.5480 - loss: 0.6839 - val_accuracy: 0.6639 - val_loss: 0.6143
Epoch 2/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 588ms/step - accuracy: 0.6677 - loss: 0.6086
Epoch 2: val_accuracy improved from 0.66387 to 0.68166, saving model to best_from_scratch.keras
251/251 ━━━━━━━━━━━━━━━━━━━━ 159s 631ms/step - accuracy: 0.6677 - loss: 0.6086 - val_accuracy: 0.6817 - val_loss: 0.5934
Epoch 3/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 587ms/step - accuracy: 0.7070 - loss: 0.5660
Epoch 3: val_accuracy improved from 0.68166 to 0.74296, saving model to best_from_scratch.keras
251/251 ━━━━━━━━━━━━━━━━━━━━ 159s 631ms/step - accuracy: 0.7070 - loss: 0.5660 - val_accuracy: 0.7430 - val_loss: 0.5242
Epoch 4/15
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 588ms/step - accuracy: 0.7252 - loss: 0.5412


In [11]:
print("Final Training Accuracy:", history.history['accuracy'][-1])
print("Final Validation Accuracy:", history.history['val_accuracy'][-1])


Final Training Accuracy: 0.8241099119186401
Final Validation Accuracy: 0.8344043493270874


In [12]:
history_extra = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=[checkpoint]   # keeps saving the best version
)


Epoch 1/3
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 557ms/step - accuracy: 0.8295 - loss: 0.3713
Epoch 1: val_accuracy did not improve from 0.83440
251/251 ━━━━━━━━━━━━━━━━━━━━ 150s 596ms/step - accuracy: 0.8295 - loss: 0.3714 - val_accuracy: 0.8344 - val_loss: 0.3834
Epoch 2/3
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 575ms/step - accuracy: 0.8305 - loss: 0.3735
Epoch 2: val_accuracy improved from 0.83440 to 0.84182, saving model to best_from_scratch.keras
251/251 ━━━━━━━━━━━━━━━━━━━━ 155s 618ms/step - accuracy: 0.8305 - loss: 0.3735 - val_accuracy: 0.8418 - val_loss: 0.3540
Epoch 3/3
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 578ms/step - accuracy: 0.8344 - loss: 0.3594
Epoch 3: val_accuracy did not improve from 0.84182
251/251 ━━━━━━━━━━━━━━━━━━━━ 155s 616ms/step - accuracy: 0.8344 - loss: 0.3594 - val_accuracy: 0.8384 - val_loss: 0.3804


In [14]:
print("Training Accuracy:", history_extra.history['accuracy'][-1])
print("Validation Accuracy:", history_extra.history['val_accuracy'][-1])


Training Accuracy: 0.8356027603149414
Validation Accuracy: 0.8383588790893555


In [16]:
# Combine training accuracy
train_acc = history.history['accuracy'] + history_extra.history['accuracy']

# Combine validation accuracy
val_acc = history.history['val_accuracy'] + history_extra.history['val_accuracy']

# Print all epochs together
for i in range(len(train_acc)):
    print(f"Epoch {i+1}: "
          f"Train Acc = {train_acc[i]:.4f}, "
          f"Val Acc = {val_acc[i]:.4f}")


Epoch 1: Train Acc = 0.5958, Val Acc = 0.6639
Epoch 2: Train Acc = 0.6776, Val Acc = 0.6817
Epoch 3: Train Acc = 0.7079, Val Acc = 0.7430
Epoch 4: Train Acc = 0.7338, Val Acc = 0.7538
Epoch 5: Train Acc = 0.7515, Val Acc = 0.7405
Epoch 6: Train Acc = 0.7553, Val Acc = 0.7771
Epoch 7: Train Acc = 0.7640, Val Acc = 0.7860
Epoch 8: Train Acc = 0.7808, Val Acc = 0.8013
Epoch 9: Train Acc = 0.7955, Val Acc = 0.8122
Epoch 10: Train Acc = 0.7963, Val Acc = 0.8166
Epoch 11: Train Acc = 0.8027, Val Acc = 0.8240
Epoch 12: Train Acc = 0.8167, Val Acc = 0.7642
Epoch 13: Train Acc = 0.8124, Val Acc = 0.8230
Epoch 14: Train Acc = 0.8257, Val Acc = 0.8309
Epoch 15: Train Acc = 0.8241, Val Acc = 0.8344
Epoch 16: Train Acc = 0.8275, Val Acc = 0.8344
Epoch 17: Train Acc = 0.8335, Val Acc = 0.8418
Epoch 18: Train Acc = 0.8356, Val Acc = 0.8384


In [17]:
for i in range(1, len(val_acc)):
    diff = val_acc[i] - val_acc[i-1]
    print(f"Epoch {i} → {i+1}: Val Acc Change = {diff:.4f}")


Epoch 1 → 2: Val Acc Change = 0.0178
Epoch 2 → 3: Val Acc Change = 0.0613
Epoch 3 → 4: Val Acc Change = 0.0109
Epoch 4 → 5: Val Acc Change = -0.0133
Epoch 5 → 6: Val Acc Change = 0.0366
Epoch 6 → 7: Val Acc Change = 0.0089
Epoch 7 → 8: Val Acc Change = 0.0153
Epoch 8 → 9: Val Acc Change = 0.0109
Epoch 9 → 10: Val Acc Change = 0.0044
Epoch 10 → 11: Val Acc Change = 0.0074
Epoch 11 → 12: Val Acc Change = -0.0598
Epoch 12 → 13: Val Acc Change = 0.0588
Epoch 13 → 14: Val Acc Change = 0.0079
Epoch 14 → 15: Val Acc Change = 0.0035
Epoch 15 → 16: Val Acc Change = 0.0000
Epoch 16 → 17: Val Acc Change = 0.0074
Epoch 17 → 18: Val Acc Change = -0.0035


In [18]:
import os
print(os.path.getsize("best_from_scratch.keras") / (1024*1024), "MB")


112.50110816955566 MB
